In [3]:
# EXP-053 — RESEARCH INVENTORY
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import re
import json
from datetime import datetime

BASE_CANDIDATES = [
    Path("astronomy_exp005_backup"),
    Path("MyDrive/astronomy"),
    Path("MyDrive")
]

# Find the project root using known research artifacts.
project_root = None
for base in BASE_CANDIDATES:
    if not base.exists():
        continue
    if list(base.rglob("exp005_all_source_groups.csv")):
        project_root = base
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the astronomy project. "
        "Expected exp005_all_source_groups.csv somewhere under MyDrive."
    )

print(f"[OK] Project root: {project_root}")

audit_root = project_root / "results" / "exp053_phase8_audit"
audit_root.mkdir(parents=True, exist_ok=True)

# Inventory research files.
patterns = [
    "exp*.csv",
    "exp*.json",
    "exp*.parquet",
    "exp*.png",
    "exp*.jpg",
    "exp*.jpeg",
    "exp*.ipynb",
    "exp*.py"
]

files_found = []
seen = set()

for pattern in patterns:
    for p in project_root.rglob(pattern):
        if not p.is_file():
            continue
        rp = p.resolve()
        if rp in seen:
            continue
        seen.add(rp)

        m = re.search(r"exp(\d{3})([A-Za-z]*)", p.name, re.I)
        if not m:
            continue

        exp_num = int(m.group(1))
        suffix = m.group(2).upper()

        stat = p.stat()

        files_found.append({
            "experiment": exp_num,
            "suffix": suffix,
            "experiment_label": f"EXP-{exp_num:03d}{suffix}",
            "filename": p.name,
            "extension": p.suffix.lower(),
            "relative_path": str(p.relative_to(project_root)),
            "absolute_path": str(p),
            "size_bytes": stat.st_size,
            "modified_time": datetime.fromtimestamp(
                stat.st_mtime
            ).isoformat(timespec="seconds")
        })

inventory = pd.DataFrame(files_found)

if inventory.empty:
    raise RuntimeError("No EXP files were discovered.")

inventory = inventory.sort_values(
    ["experiment", "suffix", "filename", "relative_path"]
).reset_index(drop=True)

inventory.to_csv(
    audit_root / "exp053_research_inventory.csv",
    index=False
)

# Experiment coverage.
covered = sorted(
    set(inventory.loc[
        inventory["experiment"].between(1, 52),
        "experiment"
    ])
)

expected = set(range(1, 53))
missing = sorted(expected - set(covered))

print("\n" + "=" * 70)
print("EXP-053 — RESEARCH INVENTORY")
print("=" * 70)

print(f"[OK] Files discovered: {len(inventory)}")
print(f"[OK] Experiments represented: {len(covered)}/52")

if missing:
    print(f"[WARN] Missing experiment IDs: {missing}")
else:
    print("[OK] EXP-001 through EXP-052 all represented")

print("\nFILES BY EXPERIMENT")
counts = (
    inventory[inventory["experiment"].between(1, 52)]
    .groupby("experiment")
    .size()
)

for n in range(1, 53):
    print(f"EXP-{n:03d}: {int(counts.get(n, 0))} files")

print("\nIMPORTANT RESEARCH ARTIFACTS")

important = [
    "exp005_all_source_groups.csv",
    "exp005_region_detections.csv",
    "exp019D_corrected_validator_results.csv",
    "exp032_leave_one_out.csv",
    "exp033_influential_observations.csv",
    "exp034F_comparison.csv"
]

for name in important:
    matches = inventory[inventory["filename"].eq(name)]
    if len(matches):
        print(f"[FOUND] {name}: {len(matches)}")
    else:
        print(f"[MISSING] {name}")

print(f"\n[OK] Inventory saved:")
print(audit_root / "exp053_research_inventory.csv")

print("\nSTATUS: EXP-053 COMPLETE — INVENTORY ONLY")

Mounted at /content/drive
[OK] Project root: astronomy_exp005_backup

EXP-053 — RESEARCH INVENTORY
[OK] Files discovered: 228
[OK] Experiments represented: 48/52
[WARN] Missing experiment IDs: [1, 2, 3, 4]

FILES BY EXPERIMENT
EXP-001: 0 files
EXP-002: 0 files
EXP-003: 0 files
EXP-004: 0 files
EXP-005: 6 files
EXP-006: 3 files
EXP-007: 2 files
EXP-008: 4 files
EXP-009: 4 files
EXP-010: 2 files
EXP-011: 2 files
EXP-012: 2 files
EXP-013: 2 files
EXP-014: 2 files
EXP-015: 2 files
EXP-016: 2 files
EXP-017: 3 files
EXP-018: 2 files
EXP-019: 8 files
EXP-020: 7 files
EXP-021: 8 files
EXP-022: 13 files
EXP-023: 4 files
EXP-024: 3 files
EXP-025: 3 files
EXP-026: 3 files
EXP-027: 3 files
EXP-028: 2 files
EXP-029: 2 files
EXP-030: 2 files
EXP-031: 3 files
EXP-032: 4 files
EXP-033: 4 files
EXP-034: 15 files
EXP-035: 3 files
EXP-036: 2 files
EXP-037: 1 files
EXP-038: 31 files
EXP-039: 1 files
EXP-040: 1 files
EXP-041: 1 files
EXP-042: 3 files
EXP-043: 5 files
EXP-044: 5 files
EXP-045: 2 files
EXP-0

In [4]:
# EXP-054 — NUMERICAL CONSISTENCY AUDIT
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

project_root = Path("astronomy_exp005_backup")
audit_root = project_root / "results" / "exp053_phase8_audit"
audit_root.mkdir(parents=True, exist_ok=True)

inventory_path = audit_root / "exp053_research_inventory.csv"
if not inventory_path.exists():
    raise FileNotFoundError("Run EXP-053 first.")

inventory = pd.read_csv(inventory_path)

def find_file(name):
    rows = inventory[inventory["filename"].eq(name)]
    if rows.empty:
        return None
    return Path(rows.iloc[0]["absolute_path"])

def find_exp(exp_num, keywords=None):
    rows = inventory[inventory["experiment"].eq(exp_num)].copy()
    if keywords:
        mask = rows["filename"].str.lower().apply(
            lambda x: all(k.lower() in x for k in keywords)
        )
        rows = rows[mask]
    return rows

def read_csv(name):
    p = find_file(name)
    if p is None:
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[WARN] Could not read {name}: {e}")
        return None

audit_rows = []
candidate_ids = [1974, 1978, 233, 1976, 234, 1979]

# ------------------------------------------------------------
# 1. Core authoritative artifacts
# ------------------------------------------------------------

core_files = [
    "exp005_all_source_groups.csv",
    "exp005_region_detections.csv",
    "exp019D_corrected_validator_results.csv",
    "exp032_leave_one_out.csv",
    "exp033_influential_observations.csv",
    "exp034F_comparison.csv"
]

print("=" * 70)
print("EXP-054 — NUMERICAL CONSISTENCY AUDIT")
print("=" * 70)

print("\nCORE ARTIFACT STATUS")

for name in core_files:
    p = find_file(name)
    status = "FOUND" if p else "MISSING"
    print(f"[{status}] {name}")

# ------------------------------------------------------------
# 2. Load source-level authoritative data
# ------------------------------------------------------------

groups = read_csv("exp005_all_source_groups.csv")
validator = read_csv("exp019D_corrected_validator_results.csv")
loo = read_csv("exp032_leave_one_out.csv")
image = read_csv("exp034F_comparison.csv")

if groups is None or validator is None or loo is None:
    raise RuntimeError("One or more required authoritative files are missing.")

# Normalize IDs.
for df in [groups, validator, loo]:
    if "group_id" in df.columns:
        df["group_id"] = pd.to_numeric(df["group_id"], errors="coerce").astype("Int64")

# ------------------------------------------------------------
# 3. Source-level cross-check: EXP-005 vs EXP-019D
# ------------------------------------------------------------

print("\nSOURCE-LEVEL CROSS-CHECK")

for gid in candidate_ids:
    g = groups[groups["group_id"].eq(gid)]
    v = validator[validator["group_id"].eq(gid)]

    if g.empty:
        print(f"[WARN] {gid}: missing from EXP-005")
        continue

    if v.empty:
        print(f"[WARN] {gid}: missing from EXP-019D")
        continue

    g = g.iloc[0]
    v = v.iloc[0]

    fields = [
        ("n_observations", "n_obs"),
        ("w1_reduced_chi2", "w1_reduced_chi2_recomputed"),
        ("median_w1_snr", "median_w1_snr_recomputed"),
        ("good_quality_fraction", "quality_fraction_correct")
    ]

    for left, right in fields:
        if left not in g or right not in v:
            continue

        a = pd.to_numeric(pd.Series([g[left]]), errors="coerce").iloc[0]
        b = pd.to_numeric(pd.Series([v[right]]), errors="coerce").iloc[0]

        if pd.isna(a) or pd.isna(b):
            continue

        diff = float(a - b)
        rel = float(diff / b) if b != 0 else np.nan

        audit_rows.append({
            "source": "EXP-005_vs_EXP-019D",
            "group_id": int(gid),
            "metric": left,
            "value_a": float(a),
            "value_b": float(b),
            "difference": diff,
            "relative_difference": rel,
            "status": "MATCH" if np.isclose(a, b, rtol=1e-8, atol=1e-10) else "DIFF"
        })

        print(
            f"{gid} {left}: "
            f"EXP005={a:.8g} EXP019D={b:.8g} "
            f"diff={diff:.3g}"
        )

# ------------------------------------------------------------
# 4. LOO audit
# ------------------------------------------------------------

print("\nLOO STABILITY AUDIT")

if not loo.empty:
    if "survivor" in loo.columns:
        loo["survivor"] = pd.to_numeric(
            loo["survivor"], errors="coerce"
        ).astype("Int64")

    for gid in [233, 1974, 1976, 1978]:
        x = loo[loo["survivor"].eq(gid)]

        if x.empty:
            print(f"[WARN] {gid}: no LOO rows")
            continue

        chi_col = None
        for c in ["leave_one_out_chi2", "loo_chi2"]:
            if c in x.columns:
                chi_col = c
                break

        if chi_col is None:
            print(f"[WARN] {gid}: no LOO chi2 column")
            continue

        vals = pd.to_numeric(x[chi_col], errors="coerce").dropna()

        if vals.empty:
            continue

        print(
            f"{gid}: n={len(vals)} "
            f"min={vals.min():.6f} "
            f"max={vals.max():.6f} "
            f"frac>=2={(vals >= 2).mean():.6f}"
        )

# ------------------------------------------------------------
# 5. Image evidence audit
# ------------------------------------------------------------

print("\nIMAGE EVIDENCE AUDIT")

if image is not None and not image.empty:
    for gid in [1974, 1978]:
        x = image[pd.to_numeric(image["survivor"], errors="coerce").eq(gid)]

        if x.empty:
            print(f"[WARN] {gid}: no image comparison")
            continue

        row = x.iloc[0]

        print(
            f"{gid}: "
            f"flux_ratio={row.get('flux_ratio', np.nan):.6f} "
            f"SNR_ratio="
            f"{(
                row.get('influential_median_snr', np.nan) /
                row.get('normal_median_snr', np.nan)
            ) if pd.notna(row.get('normal_median_snr', np.nan)) and row.get('normal_median_snr', 0) != 0 else np.nan:.6f}"
        )

# ------------------------------------------------------------
# 6. Search Phase-VII outputs automatically
# ------------------------------------------------------------

print("\nPHASE-VII ARTIFACT DISCOVERY")

phase7 = inventory[inventory["experiment"].between(49, 52)].copy()

for _, r in phase7.iterrows():
    print(
        f"{r['experiment_label']:8s} "
        f"{r['filename']}"
    )

# Identify candidate-containing CSVs from EXP-049 to EXP-052.
phase7_tables = []

for _, r in phase7[phase7["extension"].eq(".csv")].iterrows():
    p = Path(r["absolute_path"])
    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    if "group_id" not in df.columns:
        continue

    ids = pd.to_numeric(df["group_id"], errors="coerce")
    hits = df[ids.isin(candidate_ids)]

    if not hits.empty:
        phase7_tables.append({
            "experiment": r["experiment_label"],
            "filename": r["filename"],
            "path": str(p),
            "rows": len(df),
            "candidate_rows": len(hits),
            "columns": len(df.columns)
        })

print("\nCANDIDATE-CONTAINING PHASE-VII TABLES")

for x in phase7_tables:
    print(
        f"[FOUND] {x['experiment']} | "
        f"{x['filename']} | "
        f"rows={x['rows']} candidate_rows={x['candidate_rows']}"
    )

# ------------------------------------------------------------
# 7. Explicit known discrepancy checks
# ------------------------------------------------------------

print("\nKNOWN DISCREPANCY CHECK")

expected_discrepancies = {
    1974: {
        "earlier_chi2": 3.026153,
        "phase7_chi2": 3.047284
    },
    1978: {
        "earlier_chi2": 3.939207,
        "phase7_chi2": 4.140438
    }
}

discrepancy_rows = []

for gid, vals in expected_discrepancies.items():
    diff = vals["phase7_chi2"] - vals["earlier_chi2"]

    discrepancy_rows.append({
        "group_id": gid,
        "earlier_authoritative_context": vals["earlier_chi2"],
        "phase7_recomputed_context": vals["phase7_chi2"],
        "difference": diff,
        "status": "DOCUMENT_METHOD_DIFFERENCE"
    })

    print(
        f"{gid}: earlier={vals['earlier_chi2']:.6f}, "
        f"PhaseVII={vals['phase7_chi2']:.6f}, "
        f"delta={diff:+.6f}"
    )

# ------------------------------------------------------------
# 8. Save numerical audit
# ------------------------------------------------------------

comparison_df = pd.DataFrame(audit_rows)
discrepancy_df = pd.DataFrame(discrepancy_rows)
phase7_df = pd.DataFrame(phase7_tables)

comparison_df.to_csv(
    audit_root / "exp054_core_numerical_comparison.csv",
    index=False
)

discrepancy_df.to_csv(
    audit_root / "exp054_known_discrepancies.csv",
    index=False
)

phase7_df.to_csv(
    audit_root / "exp054_phase7_artifact_map.csv",
    index=False
)

summary = {
    "experiment": "EXP-054",
    "core_files_found": int(sum(find_file(x) is not None for x in core_files)),
    "core_files_expected": len(core_files),
    "candidate_ids_reviewed": candidate_ids,
    "known_discrepancies": 2,
    "interpretation": (
        "Saved analyses are numerically traceable, but "
        "method/version differences between earlier and Phase-VII "
        "chi-square calculations must be explicitly documented. "
        "The Phase-VII calculation should be used consistently for "
        "Phase-VII claims."
    ),
    "status": "PASS_WITH_DOCUMENTED_DISCREPANCIES"
}

with open(audit_root / "exp054_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("STATUS: PASS_WITH_DOCUMENTED_DISCREPANCIES")
print("=" * 70)
print("No numbers were silently overwritten.")
print("The 1974/1978 chi-square differences are preserved for the paper audit.")
print(f"Saved to: {audit_root}")

EXP-054 — NUMERICAL CONSISTENCY AUDIT

CORE ARTIFACT STATUS
[FOUND] exp005_all_source_groups.csv
[FOUND] exp005_region_detections.csv
[FOUND] exp019D_corrected_validator_results.csv
[FOUND] exp032_leave_one_out.csv
[FOUND] exp033_influential_observations.csv
[FOUND] exp034F_comparison.csv

SOURCE-LEVEL CROSS-CHECK
1974 n_observations: EXP005=25 EXP019D=25 diff=0
1974 w1_reduced_chi2: EXP005=3.0261527 EXP019D=3.0261527 diff=-1.78e-14
1974 median_w1_snr: EXP005=22.5 EXP019D=22.61875 diff=-0.119
1974 good_quality_fraction: EXP005=1 EXP019D=1 diff=0
1978 n_observations: EXP005=28 EXP019D=28 diff=0
1978 w1_reduced_chi2: EXP005=3.9392068 EXP019D=3.9392068 diff=1.29e-14
1978 median_w1_snr: EXP005=16.45 EXP019D=16.453777 diff=-0.00378
1978 good_quality_fraction: EXP005=1 EXP019D=1 diff=0
233 n_observations: EXP005=22 EXP019D=22 diff=0
233 w1_reduced_chi2: EXP005=2.1184748 EXP019D=2.1184748 diff=-1.69e-14
233 median_w1_snr: EXP005=34.85 EXP019D=35.022581 diff=-0.173
233 good_quality_fraction: E

In [5]:
# EXP-055 — CLAIM / EVIDENCE AUDIT
from pathlib import Path
import pandas as pd
import json

project_root = Path("astronomy_exp005_backup")
audit_root = project_root / "results" / "exp053_phase8_audit"
audit_root.mkdir(parents=True, exist_ok=True)

inventory = pd.read_csv(audit_root / "exp053_research_inventory.csv")

def exists(name):
    return bool((inventory["filename"] == name).any())

claims = [
    {
        "claim_id": "C01",
        "claim": "The corrected internal validator identified six positive source groups among 82 unique evaluation groups.",
        "evidence": "EXP-019D corrected validator results",
        "artifact": "exp019D_corrected_validator_results.csv",
        "strength": "STRONG_INTERNAL",
        "limitation": "This is an internal field comparison, not an independent false-positive rate.",
        "allowed_wording": "Six of 82 unique evaluation groups met the baseline trigger criteria.",
        "forbidden_overclaim": "The false-positive rate is 7.32%."
    },
    {
        "claim_id": "C02",
        "claim": "Sources 1974 and 1978 show persistent W1 excess scatter under the Phase-III validator.",
        "evidence": "EXP-019D + EXP-032",
        "artifact": "exp019D_corrected_validator_results.csv; exp032_leave_one_out.csv",
        "strength": "STRONG_INTERNAL",
        "limitation": "Persistence of the statistical anomaly does not establish astrophysical variability.",
        "allowed_wording": "Both Tier-1 candidates retain elevated W1 reduced chi-square under leave-one-out testing.",
        "forbidden_overclaim": "Both sources are confirmed variable stars."
    },
    {
        "claim_id": "C03",
        "claim": "1974 and 1978 remain unusual relative to matched observing-condition controls.",
        "evidence": "EXP-050",
        "artifact": "exp050_matched_control_comparison.csv; exp050_final_summary.csv",
        "strength": "STRONG_COMPARATIVE",
        "limitation": "The matched-control sample is small and field-specific.",
        "allowed_wording": "Both Tier-1 candidates remained unusual relative to the matched controls tested.",
        "forbidden_overclaim": "The anomalies cannot be instrumental."
    },
    {
        "claim_id": "C04",
        "claim": "Simple neighbor-tracking does not reproduce the candidate behavior.",
        "evidence": "EXP-049",
        "artifact": "exp049_blending_summary.csv; exp049_candidate_summary.csv",
        "strength": "MODERATE",
        "limitation": "A neighbor need not vary in the same way for aperture contamination to occur.",
        "allowed_wording": "Neighbor tracking provided weak or inconclusive support for a simple blending explanation.",
        "forbidden_overclaim": "Blending has been ruled out."
    },
    {
        "claim_id": "C05",
        "claim": "1978 has a nearby NEOWISE source at approximately 11.29 arcsec.",
        "evidence": "EXP-043 + EXP-049",
        "artifact": "exp049_local_neighbors_30arcsec.csv",
        "strength": "DIRECT",
        "limitation": "Proximity alone does not establish photometric contamination.",
        "allowed_wording": "A nearby NEOWISE source lies approximately 11.29 arcsec from source 1978.",
        "forbidden_overclaim": "The neighboring source contaminates 1978."
    },
    {
        "claim_id": "C06",
        "claim": "Image analysis supports the presence of a source-like signal at the target positions.",
        "evidence": "EXP-034F",
        "artifact": "exp034F_comparison.csv",
        "strength": "MODERATE",
        "limitation": "Only a small number of image epochs were compared; source-like morphology does not prove variability.",
        "allowed_wording": "The image-level measurements are consistent with source-like detections at the target positions.",
        "forbidden_overclaim": "Image photometry confirms astrophysical variability."
    },
    {
        "claim_id": "C07",
        "claim": "No valid SIMBAD object was returned for 1974 or 1978 in the tested query.",
        "evidence": "EXP-043 + EXP-051",
        "artifact": "exp051_external_known_class.csv",
        "strength": "DIRECT_QUERY_RESULT",
        "limitation": "Gaia remained unresolved because the query failed; absence from the tested SIMBAD query is not absence from all astronomical catalogs.",
        "allowed_wording": "No valid SIMBAD object was returned for either Tier-1 candidate in the tested query.",
        "forbidden_overclaim": "The candidates have no known astronomical identity."
    },
    {
        "claim_id": "C08",
        "claim": "No known SIMBAD class was identified for the Tier-1 candidates in the tested catalog query.",
        "evidence": "EXP-051",
        "artifact": "exp051_external_known_class.csv",
        "strength": "LIMITED_EXTERNAL",
        "limitation": "This does not establish that the objects are members of a new astrophysical class.",
        "allowed_wording": "No known SIMBAD class was identified in the tested catalog query.",
        "forbidden_overclaim": "The objects represent a new class of astronomical source."
    },
    {
        "claim_id": "C09",
        "claim": "1974 and 1978 survived the Phase-VII quantitative challenges performed in this project.",
        "evidence": "EXP-049 + EXP-050 + EXP-051",
        "artifact": "exp049_candidate_summary.csv; exp050_final_summary.csv; exp051_external_known_class.csv",
        "strength": "MODERATE_TO_STRONG",
        "limitation": "Several external/systematic tests remain incomplete, especially Gaia identity and complete image-mask analysis.",
        "allowed_wording": "The candidates remained unusual after the quantitative neighbor and matched-control challenges performed here.",
        "forbidden_overclaim": "The candidates have been proven astrophysical."
    },
    {
        "claim_id": "C10",
        "claim": "The final project does not confirm either candidate as an astrophysical variable or new class.",
        "evidence": "EXP-052",
        "artifact": "exp052_final_candidate_dossiers.csv; exp052_phase7_synthesis.csv",
        "strength": "STRONG_CONSERVATIVE",
        "limitation": "Confirmation requires external or observational follow-up beyond the current analysis.",
        "allowed_wording": "Neither candidate is confirmed; both remain high-priority follow-up candidates.",
        "forbidden_overclaim": "The study discovered two new astronomical objects."
    },
    {
        "claim_id": "C11",
        "claim": "The unsupervised ML component provides population-level triage rather than confirmation.",
        "evidence": "EXP-038 + EXP-048",
        "artifact": "EXP-038/EXP-048 outputs",
        "strength": "STRONG_METHOD",
        "limitation": "The reference population is small and field-specific, and evaluation labels are not ground truth.",
        "allowed_wording": "Unsupervised models were used for descriptive novelty triage and robustness analysis.",
        "forbidden_overclaim": "The ML model detects new astronomical classes."
    },
    {
        "claim_id": "C12",
        "claim": "Unknown external information is retained as unknown rather than interpreted as negative evidence.",
        "evidence": "EXP-043 + EXP-051 + project methodology",
        "artifact": "exp051_external_known_class.csv",
        "strength": "METHOD",
        "limitation": "External catalog coverage remains incomplete.",
        "allowed_wording": "Failed or unavailable external queries were retained as UNKNOWN rather than converted to negative labels.",
        "forbidden_overclaim": "The candidates were absent from all external catalogs."
    }
]

df = pd.DataFrame(claims)

# Verify referenced artifacts where possible.
def artifact_status(x):
    parts = [p.strip() for p in x.split(";")]
    found = 0
    for p in parts:
        if p.startswith("EXP-038/"):
            found += 1
        elif exists(p):
            found += 1
    return "FOUND" if found else "NOT_FOUND"

df["artifact_status"] = df["artifact"].apply(artifact_status)

# Add explicit audit flags.
df["paper_safe"] = df["strength"].isin([
    "STRONG_INTERNAL",
    "STRONG_COMPARATIVE",
    "STRONG_CONSERVATIVE",
    "STRONG_METHOD",
    "DIRECT",
    "DIRECT_QUERY_RESULT",
    "MODERATE",
    "MODERATE_COMPARATIVE",
    "MODERATE_TO_STRONG",
    "LIMITED_EXTERNAL",
    "METHOD"
])

df["requires_caution"] = df["strength"].isin([
    "LIMITED_EXTERNAL",
    "MODERATE",
    "MODERATE_TO_STRONG"
])

print("=" * 70)
print("EXP-055 — CLAIM / EVIDENCE AUDIT")
print("=" * 70)

print(f"[OK] Claims audited: {len(df)}")
print(f"[OK] Paper-safe claims: {int(df['paper_safe'].sum())}")
print(f"[INFO] Claims requiring explicit limitation: {int(df['requires_caution'].sum())}")

print("\nCLAIM STATUS")

for _, r in df.iterrows():
    marker = "OK" if r["paper_safe"] else "WARN"
    print(
        f"[{marker}] {r['claim_id']} | "
        f"{r['strength']} | "
        f"{r['claim']}"
    )

print("\nCRITICAL LIMITATIONS")

limitations = [
    "The 6/82 trigger rate is NOT a false-positive rate.",
    "The matched-control comparison is small and field-specific.",
    "Neighbor tracking does NOT rule out blending.",
    "No SIMBAD match does NOT mean no astronomical identity.",
    "Gaia query failure remains UNKNOWN, not negative evidence.",
    "Image source-like appearance does NOT prove astrophysical variability.",
    "The ML component performs triage/novelty analysis, not discovery confirmation.",
    "The current analysis does not establish a new astronomical class.",
    "External/observational confirmation has not yet occurred."
]

for x in limitations:
    print(f"- {x}")

df.to_csv(
    audit_root / "exp055_claim_evidence_audit.csv",
    index=False
)

summary = {
    "experiment": "EXP-055",
    "claims_audited": len(df),
    "paper_safe_claims": int(df["paper_safe"].sum()),
    "claims_requiring_caution": int(df["requires_caution"].sum()),
    "core_conclusion": (
        "The project supports a conservative claim that sources 1974 and 1978 "
        "remain persistent W1-excess-scatter candidates after the tested "
        "quantitative challenges, but neither is confirmed as an astrophysical "
        "variable or new source class."
    ),
    "major_unknowns": [
        "Gaia identity/classification",
        "complete image-mask audit",
        "independent external validation",
        "observational follow-up"
    ],
    "status": "PASS — CLAIMS CONSERVATIVELY BOUNDED"
}

with open(audit_root / "exp055_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("STATUS: PASS — CLAIMS CONSERVATIVELY BOUNDED")
print("=" * 70)
print(f"Saved: {audit_root / 'exp055_claim_evidence_audit.csv'}")

EXP-055 — CLAIM / EVIDENCE AUDIT
[OK] Claims audited: 12
[OK] Paper-safe claims: 12
[INFO] Claims requiring explicit limitation: 4

CLAIM STATUS
[OK] C01 | STRONG_INTERNAL | The corrected internal validator identified six positive source groups among 82 unique evaluation groups.
[OK] C02 | STRONG_INTERNAL | Sources 1974 and 1978 show persistent W1 excess scatter under the Phase-III validator.
[OK] C03 | STRONG_COMPARATIVE | 1974 and 1978 remain unusual relative to matched observing-condition controls.
[OK] C04 | MODERATE | Simple neighbor-tracking does not reproduce the candidate behavior.
[OK] C05 | DIRECT | 1978 has a nearby NEOWISE source at approximately 11.29 arcsec.
[OK] C06 | MODERATE | Image analysis supports the presence of a source-like signal at the target positions.
[OK] C07 | DIRECT_QUERY_RESULT | No valid SIMBAD object was returned for 1974 or 1978 in the tested query.
[OK] C08 | LIMITED_EXTERNAL | No known SIMBAD class was identified for the Tier-1 candidates in the test

In [6]:
# EXP-056 — FINAL CANDIDATE DOSSIERS + ANALYSIS FREEZE PACKAGE
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
from datetime import datetime, timezone

project_root = Path("astronomy_exp005_backup")
audit_root = project_root / "results" / "exp053_phase8_audit"
freeze_root = project_root / "results" / "exp056_phase8_freeze"
freeze_root.mkdir(parents=True, exist_ok=True)

inventory = pd.read_csv(audit_root / "exp053_research_inventory.csv")

def locate(name):
    x = inventory[inventory["filename"].eq(name)]
    if x.empty:
        return None
    return Path(x.iloc[0]["absolute_path"])

def load(name):
    p = locate(name)
    if p is None:
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[WARN] {name}: {e}")
        return None

def num(v):
    x = pd.to_numeric(pd.Series([v]), errors="coerce").iloc[0]
    return float(x) if pd.notna(x) else np.nan

def getrow(df, gid, idcol="group_id"):
    if df is None or df.empty or idcol not in df.columns:
        return pd.DataFrame()
    ids = pd.to_numeric(df[idcol], errors="coerce")
    return df[ids.eq(gid)]

def getval(df, gid, columns, idcol="group_id"):
    x = getrow(df, gid, idcol)
    if x.empty:
        return np.nan
    r = x.iloc[0]
    for c in columns:
        if c in r.index and pd.notna(r[c]):
            return r[c]
    return np.nan

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=" * 70)
print("EXP-056 — FINAL CANDIDATE DOSSIERS + ANALYSIS FREEZE")
print("=" * 70)

# ------------------------------------------------------------
# LOAD AUTHORITATIVE ARTIFACTS
# ------------------------------------------------------------

validator = load("exp019D_corrected_validator_results.csv")
loo = load("exp032_leave_one_out.csv")
image = load("exp034F_comparison.csv")
blend = load("exp049_candidate_summary.csv")
matched = load("exp050_final_summary.csv")
external = load("exp051_external_known_class.csv")
dossiers = load("exp052_final_candidate_dossiers.csv")

required = {
    "EXP-019D": validator,
    "EXP-032": loo,
    "EXP-034F": image,
    "EXP-049": blend,
    "EXP-050": matched,
    "EXP-051": external,
    "EXP-052": dossiers
}

print("\nAUTHORITATIVE ARTIFACTS")
for name, df in required.items():
    print(f"[{'FOUND' if df is not None else 'MISSING'}] {name}")

# ------------------------------------------------------------
# FINAL CANDIDATE SET
# ------------------------------------------------------------

candidate_ids = [1978, 1974, 233, 1976, 234, 1979]

rows = []

for gid in candidate_ids:

    # Validator
    nobs = getval(
        validator, gid,
        ["n_obs", "n_observations"]
    )

    chi2 = getval(
        validator, gid,
        ["w1_reduced_chi2_recomputed", "w1_reduced_chi2"]
    )

    snr = getval(
        validator, gid,
        ["median_w1_snr_recomputed", "median_w1_snr"]
    )

    quality = getval(
        validator, gid,
        ["quality_fraction_correct", "good_quality_fraction"]
    )

    # LOO
    l = getrow(loo, gid, "survivor")

    loo_min = np.nan
    loo_max = np.nan
    loo_frac = np.nan

    if not l.empty:
        col = next(
            (
                c for c in
                ["leave_one_out_chi2", "loo_chi2"]
                if c in l.columns
            ),
            None
        )

        if col:
            vals = pd.to_numeric(
                l[col], errors="coerce"
            ).dropna()

            if len(vals):
                loo_min = float(vals.min())
                loo_max = float(vals.max())
                loo_frac = float((vals >= 2).mean())

    # Image
    image_ratio = getval(
        image, gid,
        ["flux_ratio"],
        "survivor"
    )

    image_available = pd.notna(image_ratio)

    # Phase VII matched controls
    phase7_chi2 = getval(
        matched, gid,
        ["target_w1_chi2"]
    )

    matched_verdict = getval(
        matched, gid,
        ["matched_control_verdict"]
    )

    if pd.isna(matched_verdict):
        matched_verdict = "UNKNOWN"

    # Blending
    blending = getval(
        blend, gid,
        ["blending_verdict"]
    )

    if pd.isna(blending):
        blending = "UNKNOWN"

    # External / known class
    simbad = getval(
        external, gid,
        ["simbad_valid_objects"]
    )

    gaia = getval(
        external, gid,
        ["gaia_status"]
    )

    known = getval(
        external, gid,
        ["known_class_verdict"]
    )

    if pd.isna(gaia):
        gaia = "UNKNOWN"

    if pd.isna(known):
        known = "UNKNOWN"

    # Final priority
    if gid == 1978:
        priority = "TIER_1_HIGHEST"
        status = "PERSISTENT_UNEXPLAINED_CANDIDATE"
        claim = "HIGH_PRIORITY_FOLLOW_UP_CANDIDATE"
    elif gid == 1974:
        priority = "TIER_1_HIGH"
        status = "PERSISTENT_UNEXPLAINED_CANDIDATE"
        claim = "HIGH_PRIORITY_FOLLOW_UP_CANDIDATE"
    elif gid in [233, 1976]:
        priority = "TIER_2"
        status = "REQUIRES_ADDITIONAL_CONTROLS"
        claim = "SECONDARY_CANDIDATE"
    else:
        priority = "CONTROL"
        status = "CONTROL_COMPATIBLE_SANITY_SOURCE"
        claim = "NOT_A_DISCOVERY_CLAIM"

    rows.append({
        "group_id": gid,
        "priority": priority,
        "final_status": status,
        "candidate_claim": claim,
        "n_obs": num(nobs),
        "validator_w1_chi2": num(chi2),
        "phase7_w1_chi2": num(phase7_chi2),
        "median_w1_snr": num(snr),
        "quality_fraction": num(quality),
        "loo_min_chi2": loo_min,
        "loo_max_chi2": loo_max,
        "loo_fraction_above_2": loo_frac,
        "image_flux_ratio": num(image_ratio),
        "image_evidence_available": bool(image_available),
        "blending_verdict_exp049": str(blending),
        "matched_control_verdict": str(matched_verdict),
        "simbad_valid_objects": num(simbad),
        "gaia_status": str(gaia),
        "known_class_verdict": str(known),
        "confirmation_status": "NOT_CONFIRMED"
    })

candidate_master = pd.DataFrame(rows)

# ------------------------------------------------------------
# CORRECTIONS TO MACHINE-GENERATED SYNTHESIS
# ------------------------------------------------------------

# EXP-049 is authoritative for the blending test.
candidate_master["blending_verdict_final"] = (
    candidate_master["blending_verdict_exp049"]
)

# Gaia query failures remain UNKNOWN.
candidate_master.loc[
    candidate_master["gaia_status"]
    .str.contains("FAILURE", case=False, na=False),
    "gaia_status"
] = "UNKNOWN_QUERY_FAILURE"

# No Gaia result is ever converted to a negative.
candidate_master.loc[
    candidate_master["gaia_status"].isin(["nan", "None"]),
    "gaia_status"
] = "UNKNOWN"

# Conservative scientific interpretation.
def interpretation(gid):
    if gid == 1978:
        return (
            "Highest-priority candidate. Persistent W1 excess scatter "
            "survives leave-one-out and matched-control testing. "
            "Simple neighbor tracking is weak/inconclusive. "
            "External identity remains incomplete."
        )
    if gid == 1974:
        return (
            "High-priority candidate. Persistent W1 excess scatter "
            "survives leave-one-out and matched-control testing. "
            "Simple neighbor tracking is weak/inconclusive. "
            "External identity remains incomplete."
        )
    if gid in [233, 1976]:
        return (
            "Secondary candidate requiring additional controls before "
            "any stronger interpretation."
        )
    return (
        "Control-compatible sanity source retained as a check against "
        "over-selection by the pipeline."
    )

candidate_master["scientific_interpretation"] = (
    candidate_master["group_id"].apply(interpretation)
)

candidate_master.to_csv(
    freeze_root / "exp056_candidate_master.csv",
    index=False
)

print("\nFINAL CANDIDATE MASTER")
print(
    candidate_master[
        [
            "group_id",
            "priority",
            "validator_w1_chi2",
            "phase7_w1_chi2",
            "loo_min_chi2",
            "matched_control_verdict",
            "blending_verdict_final",
            "confirmation_status"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# EXPERIMENT PROVENANCE
# ------------------------------------------------------------

provenance_rows = [
    ["EXP-019D","Corrected validator",
     "Baseline source-level trigger","exp019D_corrected_validator_results.csv",
     "AUTHORITATIVE"],
    ["EXP-032","Leave-one-out robustness",
     "Persistence under observation removal","exp032_leave_one_out.csv",
     "AUTHORITATIVE"],
    ["EXP-034F","Image comparison",
     "Source-like image evidence","exp034F_comparison.csv",
     "SUPPORTING"],
    ["EXP-038","Unsupervised ML",
     "Novelty/triage analysis","EXP-038 outputs",
     "METHOD_SUPPORT"],
    ["EXP-048","Population robustness",
     "Reference-population novelty validation","EXP-048 outputs",
     "METHOD_SUPPORT"],
    ["EXP-049","Blending challenge",
     "Quantitative neighbor tracking","exp049_candidate_summary.csv",
     "AUTHORITATIVE_PHASE_VII"],
    ["EXP-050","Matched-control challenge",
     "Observing-condition comparison","exp050_final_summary.csv",
     "AUTHORITATIVE_PHASE_VII"],
    ["EXP-051","External identity/class",
     "SIMBAD/Gaia/known-class status","exp051_external_known_class.csv",
     "AUTHORITATIVE_WITH_UNKNOWN_GAIA"],
    ["EXP-052","Phase-VII synthesis",
     "Final candidate prioritization","exp052_final_candidate_dossiers.csv",
     "SYNTHESIS_CORRECTED"]
]

provenance = pd.DataFrame(
    provenance_rows,
    columns=[
        "experiment",
        "role",
        "supports",
        "artifact",
        "authority_status"
    ]
)

provenance.to_csv(
    freeze_root / "exp056_experiment_provenance.csv",
    index=False
)

# ------------------------------------------------------------
# FREEZE CHECKS
# ------------------------------------------------------------

checks = []

def add_check(name, status, detail):
    checks.append({
        "check": name,
        "status": status,
        "detail": detail
    })

add_check(
    "EXP-053 inventory",
    "PASS" if (audit_root / "exp053_research_inventory.csv").exists()
    else "FAIL",
    "Research inventory exists."
)

add_check(
    "EXP-054 numerical audit",
    "PASS" if (audit_root / "exp054_summary.json").exists()
    else "FAIL",
    "Numerical consistency audit exists."
)

add_check(
    "EXP-055 claim audit",
    "PASS" if (audit_root / "exp055_claim_evidence_audit.csv").exists()
    else "FAIL",
    "Claim/evidence audit exists."
)

add_check(
    "Six final-review sources",
    "PASS" if set(candidate_master["group_id"]) == set(candidate_ids)
    else "FAIL",
    "Final review contains exactly six sources."
)

add_check(
    "Tier-1 set",
    "PASS" if set(
        candidate_master.loc[
            candidate_master["priority"].str.startswith("TIER_1"),
            "group_id"
        ]
    ) == {1974, 1978} else "FAIL",
    "Tier-1 candidates are 1974 and 1978."
)

add_check(
    "No confirmation overclaim",
    "PASS" if (
        candidate_master["confirmation_status"] == "NOT_CONFIRMED"
    ).all() else "FAIL",
    "No source is marked as confirmed."
)

add_check(
    "Blending correction",
    "PASS" if all(
        candidate_master.loc[
            candidate_master["group_id"].isin([1974,1978]),
            "blending_verdict_final"
        ].str.contains(
            "BLENDING_WEAK_OR_INCONCLUSIVE",
            na=False
        )
    ) else "WARN",
    "EXP-049 result restored into final candidate record."
)

add_check(
    "Gaia UNKNOWN preserved",
    "PASS" if all(
        candidate_master.loc[
            candidate_master["group_id"].isin([1974,1978]),
            "gaia_status"
        ] == "UNKNOWN_QUERY_FAILURE"
    ) else "WARN",
    "Gaia query failure is not treated as a negative result."
)

add_check(
    "Early EXP-001–004 gap",
    "WARN",
    "Detailed EXP-001–004 files were not discovered; no reconstruction is claimed."
)

add_check(
    "Chi-square discrepancy",
    "WARN",
    "Earlier and Phase-VII chi-square calculations differ and must be documented."
)

checks_df = pd.DataFrame(checks)

checks_df.to_csv(
    freeze_root / "exp056_freeze_checks.csv",
    index=False
)

# ------------------------------------------------------------
# SOURCE CHECKSUMS
# ------------------------------------------------------------

checksum_rows = []

for _, r in inventory.iterrows():
    p = Path(r["absolute_path"])

    # Exclude Phase-VIII generated files.
    try:
        p.relative_to(audit_root)
        continue
    except ValueError:
        pass

    try:
        p.relative_to(freeze_root)
        continue
    except ValueError:
        pass

    if not p.exists() or not p.is_file():
        continue

    checksum_rows.append({
        "experiment": r["experiment_label"],
        "filename": r["filename"],
        "relative_path": r["relative_path"],
        "size_bytes": int(p.stat().st_size),
        "sha256": sha256(p)
    })

checksums = pd.DataFrame(checksum_rows)

checksums.to_csv(
    freeze_root / "exp056_source_checksums.csv",
    index=False
)

# ------------------------------------------------------------
# FREEZE DECISION
# ------------------------------------------------------------

hard_failures = int((checks_df["status"] == "FAIL").sum())
warnings = int((checks_df["status"] == "WARN").sum())

if hard_failures > 0:
    freeze_status = "FREEZE_REVIEW_REQUIRED"
elif warnings > 0:
    freeze_status = "ANALYSIS_FREEZE_READY_WITH_DOCUMENTED_LIMITATIONS"
else:
    freeze_status = "ANALYSIS_FROZEN"

manifest = {
    "experiment": "EXP-056",
    "audit_phase": "Phase VIII",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(project_root),
    "audit_root": str(audit_root),
    "freeze_root": str(freeze_root),

    "inventory": {
        "files_discovered": int(len(inventory)),
        "experiments_represented": int(
            inventory[
                inventory["experiment"].between(1,52)
            ]["experiment"].nunique()
        ),
        "missing_early_experiments": [1,2,3,4]
    },

    "final_review": {
        "sources": candidate_ids,
        "tier1": [1974,1978],
        "tier2": [233,1976],
        "controls": [234,1979],
        "confirmed_sources": []
    },

    "known_corrections": [
        "EXP-052 blending field must use EXP-049 quantitative result.",
        "Gaia query failure remains UNKNOWN.",
        "No SIMBAD result is not equivalent to global catalog absence.",
        "Earlier and Phase-VII W1 chi-square calculations differ and must be methodologically documented."
    ],

    "major_scientific_limitations": [
        "No independent false-positive rate has been established.",
        "Matched controls are small and field-specific.",
        "Neighbor tracking does not rule out blending.",
        "Image evidence does not prove astrophysical variability.",
        "Gaia identity/classification remains unresolved.",
        "Complete external and observational confirmation is absent.",
        "Neither Tier-1 source is confirmed as an astrophysical variable or new class."
    ],

    "checks": {
        "hard_failures": hard_failures,
        "warnings": warnings
    },

    "freeze_status": freeze_status
}

with open(
    freeze_root / "ANALYSIS_FREEZE_MANIFEST.json",
    "w"
) as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# FINAL SYNTHESIS
# ------------------------------------------------------------

synthesis = {
    "project_conclusion": (
        "The analysis identifies two highest-priority candidates, "
        "NEOWISE source groups 1978 and 1974. Both show persistent "
        "W1 excess scatter and remain unusual relative to the matched "
        "controls tested. Quantitative neighbor tracking does not "
        "support a simple blending explanation, but blending and other "
        "instrumental/systematic explanations are not completely ruled out. "
        "External identity is incomplete because Gaia queries failed and "
        "no valid SIMBAD object was returned in the tested query. "
        "Neither candidate is confirmed as an astrophysical variable or "
        "new source class."
    ),

    "paper_position": (
        "The strongest defensible result is a candidate-prioritization "
        "and methodology result, not a confirmed astronomical discovery."
    ),

    "tier1_candidates": [1978,1974],
    "tier2_candidates": [233,1976],
    "sanity_controls": [234,1979],
    "confirmed_candidates": [],
    "status": freeze_status
}

with open(
    freeze_root / "exp056_phase8_synthesis.json",
    "w"
) as f:
    json.dump(synthesis, f, indent=2)

print("\n" + "=" * 70)
print("EXP-056 — FINAL AUDIT RESULT")
print("=" * 70)

print(f"Hard failures: {hard_failures}")
print(f"Documented warnings: {warnings}")
print(f"Freeze status: {freeze_status}")

print("\nFINAL SCIENTIFIC POSITION")
print("Tier 1: 1978, 1974")
print("Tier 2: 233, 1976")
print("Controls: 234, 1979")
print("Confirmed: NONE")

print("\nOUTPUTS")
for p in sorted(freeze_root.iterdir()):
    if p.is_file():
        print(f"[SAVED] {p.name}")

print("\nSTATUS:", freeze_status)

EXP-056 — FINAL CANDIDATE DOSSIERS + ANALYSIS FREEZE

AUTHORITATIVE ARTIFACTS
[FOUND] EXP-019D
[FOUND] EXP-032
[FOUND] EXP-034F
[FOUND] EXP-049
[FOUND] EXP-050
[FOUND] EXP-051
[FOUND] EXP-052

FINAL CANDIDATE MASTER
 group_id       priority  validator_w1_chi2  phase7_w1_chi2  loo_min_chi2                        matched_control_verdict        blending_verdict_final confirmation_status
     1978 TIER_1_HIGHEST           3.939207        4.140438      2.798462 CANDIDATE_UNUSUAL_RELATIVE_TO_MATCHED_CONTROLS BLENDING_WEAK_OR_INCONCLUSIVE       NOT_CONFIRMED
     1974    TIER_1_HIGH           3.026153        3.047284      2.142018 CANDIDATE_UNUSUAL_RELATIVE_TO_MATCHED_CONTROLS BLENDING_WEAK_OR_INCONCLUSIVE       NOT_CONFIRMED
      233         TIER_2           2.118475             NaN      1.653138                                        UNKNOWN                       UNKNOWN       NOT_CONFIRMED
     1976         TIER_2           2.115925             NaN      1.823840                           

In [7]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

ROOT = Path("astronomy_exp005_backup")
DEST = ROOT / "PAPER_ARCHIVE"

groups = {
    "01_FROZEN_ANALYSIS": [
        "ANALYSIS_FREEZE_MANIFEST.json",
        "exp056_candidate_master.csv",
        "exp056_experiment_provenance.csv",
        "exp056_freeze_checks.csv",
        "exp056_phase8_synthesis.json",
        "exp056_source_checksums.csv",
    ],
    "02_CORE_DATA": [
        "exp005_all_source_groups.csv",
        "exp005_region_detections.csv",
        "exp019D_corrected_validator_results.csv",
    ],
    "03_ROBUSTNESS": [
        "exp032_leave_one_out.csv",
        "exp033_influential_observations.csv",
        "exp034F_comparison.csv",
    ],
    "04_PHASE_VII": [
        "exp049_blending_summary.csv",
        "exp049_candidate_summary.csv",
        "exp049_local_neighbors_30arcsec.csv",
        "exp050_final_summary.csv",
        "exp050_frame_peers.csv",
        "exp050_matched_control_comparison.csv",
        "exp050_matched_controls.csv",
        "exp050_same_day_peers.csv",
        "exp051_external_known_class.csv",
        "exp052_final_candidate_dossiers.csv",
        "exp052_phase7_synthesis.csv",
    ],
    "05_ML": [
        "exp038A_canonical_features.csv",
        "exp038A_canonical_metadata.csv",
        "exp038B_novelty_scores.csv",
        "exp038C_novelty_profile.csv",
        "exp038D_brightness_controlled.csv",
        "exp038E_feature_family.csv",
        "exp038F_model_comparison.csv",
        "exp038G_stability.csv",
        "exp038H_evaluation.csv",
        "exp038I_feature_dependence.csv",
        "exp038J_missingness.csv",
        "exp038K_reference_population.csv",
        "exp038L_multiview_novelty.csv",
        "exp038M_candidate_evidence_profiles.csv",
        "exp038N_candidate_ranking.csv",
        "exp038O_human_investigation_shortlist.csv",
    ],
}

print("=" * 75)
print("PAPER ARCHIVE — RECURSIVE SOURCE DISCOVERY")
print("=" * 75)

if not ROOT.exists():
    raise FileNotFoundError(f"Research root not found:\n{ROOT}")

DEST.mkdir(parents=True, exist_ok=True)

# Index every file by basename
index = {}
for p in ROOT.rglob("*"):
    if p.is_file() and "PAPER_ARCHIVE" not in p.parts:
        index.setdefault(p.name, []).append(p)

copied = []
missing = []
duplicates = []

for group, filenames in groups.items():
    out = DEST / group
    out.mkdir(parents=True, exist_ok=True)

    print(f"\n[{group}]")

    for filename in filenames:
        matches = index.get(filename, [])

        if not matches:
            print(f"[MISSING] {filename}")
            missing.append((group, filename))
            continue

        if len(matches) > 1:
            duplicates.append((filename, matches))
            print(f"[MULTIPLE FOUND] {filename}")
            for m in matches:
                print(f"    {m}")

        # Prefer the most recently modified copy if duplicates exist
        src = max(matches, key=lambda x: x.stat().st_mtime)
        dst = out / filename

        shutil.copy2(src, dst)
        copied.append((group, filename, src))

        print(f"[COPIED] {filename}")
        print(f"         FROM: {src}")

print("\n" + "=" * 75)
print("FINAL VERIFICATION")
print("=" * 75)

expected = sum(len(v) for v in groups.values())

print(f"Expected: {expected}")
print(f"Copied:  {len(copied)}")
print(f"Missing: {len(missing)}")
print(f"Multiple matches: {len(duplicates)}")

if missing:
    print("\nFILES ACTUALLY NOT FOUND ANYWHERE:")
    for group, filename in missing:
        print(f"  [{group}] {filename}")

if duplicates:
    print("\nFILES WITH MULTIPLE COPIES:")
    for filename, matches in duplicates:
        print(f"\n{filename}")
        for m in matches:
            print(f"  - {m}")

print("\nARCHIVE CONTENTS:")
for group in groups:
    folder = DEST / group
    files_here = sorted(p.name for p in folder.iterdir() if p.is_file())

    print(f"\n{group}/")
    if files_here:
        for f in files_here:
            print(f"  └── {f}")
    else:
        print("  └── [empty]")

print("\n" + "=" * 75)

if not missing:
    print("[STATUS] PAPER ARCHIVE COMPLETE")
else:
    print("[STATUS] PAPER ARCHIVE COMPLETE WITH GENUINELY MISSING FILES")

print(f"[LOCATION] {DEST}")
print("=" * 75)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER ARCHIVE — RECURSIVE SOURCE DISCOVERY

[01_FROZEN_ANALYSIS]
[COPIED] ANALYSIS_FREEZE_MANIFEST.json
         FROM: astronomy_exp005_backup/results/exp056_phase8_freeze/ANALYSIS_FREEZE_MANIFEST.json
[COPIED] exp056_candidate_master.csv
         FROM: astronomy_exp005_backup/results/exp056_phase8_freeze/exp056_candidate_master.csv
[COPIED] exp056_experiment_provenance.csv
         FROM: astronomy_exp005_backup/results/exp056_phase8_freeze/exp056_experiment_provenance.csv
[COPIED] exp056_freeze_checks.csv
         FROM: astronomy_exp005_backup/results/exp056_phase8_freeze/exp056_freeze_checks.csv
[COPIED] exp056_phase8_synthesis.json
         FROM: astronomy_exp005_backup/results/exp056_phase8_freeze/exp056_phase8_synthesis.json
[COPIED] exp056_source_checksums.csv
         FROM: astronomy_exp005_backup/results/exp056_phase8_freeze/exp056_source_checksums.cs

In [8]:
from pathlib import Path

ROOT=Path("astronomy_exp005_backup")

print("ASTRONOMY PROJECT CODE INVENTORY")
print("="*70)

for p in sorted(ROOT.rglob("*")):
    if "PAPER_ARCHIVE" in p.parts:
        continue
    if p.is_file() and p.suffix.lower() in {".py",".ipynb",".sh",".yaml",".yml",".json"}:
        print(p.relative_to(ROOT))

ASTRONOMY PROJECT CODE INVENTORY
exp035_discovered_file_inventory.json
results/exp005_population_summary.json
results/exp006_reference_population_summary.json
results/exp007_external_validator_summary.json
results/exp008_expanded_external_validator_summary.json
results/exp009_external_classification_summary.json
results/exp010_control_summary.json
results/exp011_control_trigger_forensics_summary.json
results/exp012_epoch_systematics_summary.json
results/exp013_within_epoch_consistency_summary.json
results/exp014_cross_band_coherence_summary.json
results/exp015_known_variable_cross_band_summary.json
results/exp016_randomized_controls_summary.json
results/exp017_full_randomized_controls_summary.json
results/exp018_control_cross_band_summary.json
results/exp019D_quality_metric_reproducibility_summary.json
results/exp019a_threshold_sensitivity_summary.json
results/exp020_survivor_systematics_summary.json
results/exp021_scan_common_mode_summary.json
results/exp022A_summary.json
results/exp0

In [9]:
from pathlib import Path

roots=[
    Path("MyDrive"),
    Path("/content/drive/Shareddrives")
]

patterns=[
    "*astronomy*",
    "*NEOWISE*",
    "*neowise*",
    "*EXP*",
    "*exp*"
]

seen=set()

for root in roots:
    if not root.exists():
        continue
    for pattern in patterns:
        for p in root.rglob(pattern):
            if p.is_file() and p.suffix.lower() in {".ipynb",".py"}:
                s=str(p)
                if s not in seen:
                    seen.add(s)
                    print(s)

print("\nTOTAL:",len(seen))

Colab Notebooks/01_neowise_first_light_curve.ipynb
Colab Notebooks/Astronomy_Research_EXP005_Validation_Population.ipynb

TOTAL: 2


In [10]:
from pathlib import Path
import json

names=[
    "Untitled13.ipynb",
    "Untitled12.ipynb",
    "Exp 4-2.ipynb",
    "01_neowise_first_light_curve.ipynb",
    "Astronomy_Research_EXP005_Validation_Population.ipynb",
    "Cosmos4.ipynb"
]

root=Path("Colab Notebooks")

for name in names:
    p=root/name
    print("\n"+"="*80)
    print(name)
    print("="*80)

    if not p.exists():
        print("[MISSING]")
        continue

    nb=json.loads(p.read_text(encoding="utf-8"))
    cells=nb.get("cells",[])

    code=sum(c.get("cell_type")=="code" for c in cells)
    markdown=sum(c.get("cell_type")=="markdown" for c in cells)

    print("cells:",len(cells),"code:",code,"markdown:",markdown)

    for i,c in enumerate(cells):
        if c.get("cell_type")=="markdown":
            text=" ".join(c.get("source",[])).strip().replace("\n"," ")
            if text:
                print(f"[MD {i}] {text[:250]}")

        elif c.get("cell_type")=="code":
            src="".join(c.get("source",[])).strip()
            if src:
                first=src.splitlines()[0][:250]
                print(f"[CODE {i}] {first}")


Untitled13.ipynb
cells: 8 code: 8 markdown: 0
[CODE 0] # EXP-053 — RESEARCH INVENTORY
[CODE 1] # EXP-054 — NUMERICAL CONSISTENCY AUDIT
[CODE 2] # EXP-055 — CLAIM / EVIDENCE AUDIT
[CODE 3] # EXP-056 — FINAL CANDIDATE DOSSIERS + ANALYSIS FREEZE PACKAGE
[CODE 4] from google.colab import drive
[CODE 5] from pathlib import Path
[CODE 6] from pathlib import Path

Untitled12.ipynb
cells: 13 code: 13 markdown: 0
[CODE 0] # ============================================================
[CODE 1] import pandas as pd,os,json
[CODE 2] import pandas as pd,os,numpy as np
[CODE 3] import pandas as pd,os,numpy as np
[CODE 4] import pandas as pd,os,numpy as np
[CODE 5] import pandas as pd,os,numpy as np
[CODE 6] import pandas as pd,os,numpy as np
[CODE 7] import pandas as pd,os,numpy as np
[CODE 8] import pandas as pd
[CODE 9] import os
[CODE 10] import os

Exp 4-2.ipynb
cells: 13 code: 13 markdown: 0
[CODE 0] # ================================================================
[CODE 1] # =================

In [11]:
from pathlib import Path
import json,re

ROOT=Path("Colab Notebooks")

NOTEBOOKS=[
    "01_neowise_first_light_curve.ipynb",
    "Exp 4-2.ipynb",
    "Astronomy_Research_EXP005_Validation_Population.ipynb",
    "Untitled12.ipynb",
    "Cosmos4.ipynb",
    "Untitled13.ipynb",
]

print("ASTRONOMY NOTEBOOK PROVENANCE AUDIT")
print("="*80)

for name in NOTEBOOKS:
    path=ROOT/name
    print("\n"+"="*80)
    print(name)
    print("="*80)

    if not path.exists():
        print("[MISSING]")
        continue

    nb=json.loads(path.read_text(encoding="utf-8"))
    cells=nb.get("cells",[])

    full_code=[]
    experiments=[]
    installs=[]
    paths=[]
    urls=[]

    for i,cell in enumerate(cells):
        if cell.get("cell_type")!="code":
            continue

        src="".join(cell.get("source",[]))
        full_code.append(src)

        for m in re.findall(r'EXP-\d+[A-Za-z]*',src,re.I):
            experiments.append(m.upper())

        for line in src.splitlines():
            s=line.strip()

            if "pip install" in s.lower():
                installs.append(s[:180])

            if "/content/drive" in s or "MyDrive" in s:
                paths.append(s[:180])

            if "http://" in s or "https://" in s:
                urls.append(s[:180])

    text="\n".join(full_code)

    print("Cells:",len(cells))
    print("Code cells:",sum(c.get("cell_type")=="code" for c in cells))

    exps=sorted(set(experiments),
                key=lambda x:(int(re.search(r'\d+',x).group()),x))

    print("\nExperiments:")
    print(", ".join(exps) if exps else "NONE DETECTED")

    print("\nPackages/install commands:")
    if installs:
        for x in sorted(set(installs)):
            print(" ",x)
    else:
        print(" NONE DETECTED")

    print("\nDrive/path references:")
    if paths:
        for x in sorted(set(paths))[:15]:
            print(" ",x)
        if len(set(paths))>15:
            print(" ...",len(set(paths))-15,"more")
    else:
        print(" NONE DETECTED")

    print("\nExternal URLs:")
    if urls:
        for x in sorted(set(urls))[:15]:
            print(" ",x)
        if len(set(urls))>15:
            print(" ...",len(set(urls))-15,"more")
    else:
        print(" NONE DETECTED")

    # Identify likely output-writing operations
    output_patterns=[
        r'to_csv\(',
        r'to_json\(',
        r'to_parquet\(',
        r'savefig\(',
        r'json\.dump',
        r'open\(.*[\'"]w'
    ]

    output_hits=[]
    for pat in output_patterns:
        output_hits.extend(re.findall(pat,text,re.I))

    print("\nOutput-writing operations detected:",len(output_hits))

print("\n"+"="*80)
print("AUDIT COMPLETE")

ASTRONOMY NOTEBOOK PROVENANCE AUDIT

01_neowise_first_light_curve.ipynb
Cells: 34
Code cells: 34

Experiments:
EXP-001, EXP-002, EXP-002B, EXP-002C, EXP-003, EXP-004

Packages/install commands:
  pip install astropy numpy scipy matplotlib

Drive/path references:
 NONE DETECTED

External URLs:
  "https://irsa.ipac.caltech.edu"
  "https://irsa.ipac.caltech.edu/ibe/data/"
  "https://irsa.ipac.caltech.edu/ibe/search/"
  IRSA_DATA_ROOT = "https://irsa.ipac.caltech.edu/ibe/data/wise/neowiser/p1bm_frm"
  IRSA_SEARCH = "https://irsa.ipac.caltech.edu/ibe/search/wise/neowiser/p1bm_frm"
  f"https://irsa.ipac.caltech.edu/ibe/data/wise/neowiser/"

Output-writing operations detected: 5

Exp 4-2.ipynb
Cells: 13
Code cells: 13

Experiments:
EXP-001, EXP-002C, EXP-003, EXP-004

Packages/install commands:
 NONE DETECTED

Drive/path references:
  ZIP_PATH = Path("MyDrive/exp004_data.zip")

External URLs:
  "https://irsa.ipac.caltech.edu/ibe/data/"

Output-writing operations detected: 23

Astronomy_Resear